# 第 4 章 线性代数可视化

约定与题目见书中 [第 4 章](../../docs/part1/ch04-linear-algebra.md)。

**约定：** 列向量；`p_a = T_a_b @ p_b`；二维齐次 3×3；复合从右往左读。

先跑下一格。改数字后只重跑相关格；Kernel Restart 后必须从上到下再跑一遍。


In [ ]:
# %matplotlib inline   # 若图不显示，取消本行注释后重跑
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)


def rot2(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s], [s, c]], dtype=float)


def T2(theta, t):
    """二维齐次：先旋转 theta（弧度），再平移 t=(tx, ty)。"""
    T = np.eye(3)
    T[:2, :2] = rot2(theta)
    T[:2, 2] = t
    return T


def invert_T(T):
    """刚体逆：R.T 与 -R.T @ t，不要对刚体变换盲用通用求逆。"""
    R = T[:2, :2]
    t = T[:2, 2]
    inv = np.eye(3)
    inv[:2, :2] = R.T
    inv[:2, 2] = -R.T @ t
    return inv


def apply_T(T, pts):
    """pts: shape (2, N) -> (2, N)。一次矩阵乘，不要 for 每个点。"""
    ones = np.ones((1, pts.shape[1]))
    h = np.vstack([pts, ones])
    return (T @ h)[:2]


def house_xy():
    """开口向右的小房子，便于看出旋转和镜像。"""
    return np.array(
        [
            [0.0, 1.0, 1.0, 0.5, 0.0, 0.0],
            [0.0, 0.0, 0.8, 1.2, 0.8, 0.0],
        ]
    )


def draw_frame(ax, T=None, scale=0.6, name=""):
    origin = np.zeros((2, 1))
    axes = np.array([[scale, 0.0], [0.0, scale]])
    if T is not None:
        origin = apply_T(T, origin)
        axes = apply_T(T, axes) - origin
    o = origin[:, 0]
    ax.arrow(o[0], o[1], axes[0, 0], axes[1, 0], head_width=0.06, color="r", length_includes_head=True)
    ax.arrow(o[0], o[1], axes[0, 1], axes[1, 1], head_width=0.06, color="g", length_includes_head=True)
    if name:
        ax.text(o[0], o[1], " " + name)


def setup_ax(ax, title, lim=2.5):
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.3)
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.axhline(0, color="k", lw=0.4)
    ax.axvline(0, color="k", lw=0.4)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title(title)


## 1. 向量与点积

改 `a`、`b` 后重跑本格。题 1 用 `a=(3,4)`、`b=(4,-3)`。


In [ ]:
a = np.array([3.0, 4.0])
b = np.array([4.0, -3.0])

dot = a @ b
na, nb = np.linalg.norm(a), np.linalg.norm(b)
cos = np.clip(dot / (na * nb), -1.0, 1.0)
ang = np.degrees(np.arccos(cos))
proj = (dot / (na**2)) * a

print(f"a·b = {dot}")
print(f"|a|={na}, |b|={nb}, 夹角={ang:.1f} deg")
print(f"b 在 a 上的投影 = {proj}")

fig, ax = plt.subplots(figsize=(5, 5))
setup_ax(ax, "点积与投影", lim=6)
ax.arrow(0, 0, a[0], a[1], head_width=0.15, color="C0", length_includes_head=True, label="a")
ax.arrow(0, 0, b[0], b[1], head_width=0.15, color="C1", length_includes_head=True, label="b")
ax.plot([b[0], proj[0]], [b[1], proj[1]], "--", color="gray")
ax.scatter(*proj, color="C2", zorder=3, label="proj")
ax.legend()
plt.show()


## 2. 矩阵是变换；行列式是有向面积

小房子的底边、竖边应对上矩阵的两列。对题 4 的 A/B/C/D 各看一次。


In [ ]:
A = np.array([[0.0, -1.0], [1.0, 0.0]])
B = np.array([[2.0, 0.0], [0.0, 2.0]])
C = np.array([[0.0, 1.0], [1.0, 0.0]])
D = np.array([[1.0, 0.0], [0.0, -1.0]])
matrices = [("A 旋转?", A), ("B 缩放?", B), ("C 反射/置换?", C), ("D 反射?", D)]

H = house_xy()
fig, axes = plt.subplots(1, 4, figsize=(14, 3.6))
for ax, (title, M) in zip(axes, matrices):
    setup_ax(ax, f"{title}\ndet={np.linalg.det(M):.1f}", lim=2.8)
    ax.plot(H[0], H[1], color="0.7", lw=1)
    Hp = M @ H
    ax.plot(Hp[0], Hp[1], color="C3", lw=2)
    ax.arrow(0, 0, M[0, 0], M[1, 0], head_width=0.08, color="r", length_includes_head=True)
    ax.arrow(0, 0, M[0, 1], M[1, 1], head_width=0.08, color="g", length_includes_head=True)
plt.tight_layout()
plt.show()


## 3. 先转后移 ≠ 先移后转（题 5）

参考点是房子右下角附近的 `(1, 0)`。看甲、乙把整栋房子送到哪。


In [ ]:
p = np.array([1.0, 0.0])
T_jia = T2(np.pi / 2, (1.0, 0.0))  # 先转 90°，再平移 (1, 0)

R90 = np.eye(3)
R90[:2, :2] = rot2(np.pi / 2)
T_yi = R90 @ T2(0.0, (1.0, 0.0))  # 先平移 (1, 0)，再转 90°

print("甲 T =\n", T_jia, "\n点 ->", apply_T(T_jia, p.reshape(2, 1)).ravel())
print("乙 T =\n", T_yi, "\n点 ->", apply_T(T_yi, p.reshape(2, 1)).ravel())

H = house_xy()
fig, ax = plt.subplots(figsize=(5.5, 5.5))
setup_ax(ax, "甲(红) 先转后移；乙(蓝) 先移后转", lim=3)
ax.plot(H[0], H[1], color="0.5", lw=1, label="原")
Hj = apply_T(T_jia, H)
Hy = apply_T(T_yi, H)
ax.plot(Hj[0], Hj[1], color="C3", lw=2, label="甲")
ax.plot(Hy[0], Hy[1], color="C0", lw=2, label="乙")
ax.scatter([1], [0], c="k", zorder=3)
ax.legend()
plt.show()


## 4. 齐次复合与往返误差（本章硬指标）

手写 `invert_T` 必须和「先 \(T_2\) 再 \(T_1\)」的约定一致。误差应在 `1e-12` 量级。


In [ ]:
T1 = T2(0.0, (2.0, 0.0))
T2m = T2(np.pi / 2, (0.0, 0.0))
T = T1 @ T2m
p_local = np.array([[1.0], [0.0]])
print("题 6 p_w =", apply_T(T, p_local).ravel())

inv = invert_T(T)
err_hand = np.max(np.abs(inv @ T - np.eye(3)))
err_np = np.max(np.abs(np.linalg.inv(T) @ T - np.eye(3)))
print(f"手写逆  max|T^{-1}T - I| = {err_hand:.2e}")
print(f"numpy逆 max|T^{-1}T - I| = {err_np:.2e}")
assert err_hand < 1e-12

rng = np.random.default_rng(0)
errs = []
for _ in range(20):
    Tk = T2(rng.uniform(-np.pi, np.pi), rng.normal(size=2))
    errs.append(np.max(np.abs(invert_T(Tk) @ Tk - np.eye(3))))
print("随机 20 组最大往返误差", max(errs))

H = house_xy()
fig, ax = plt.subplots(figsize=(5.5, 5.5))
setup_ax(ax, "局部 → T2 → T1@T2", lim=3.5)
ax.plot(*H, color="0.6", label="local")
ax.plot(*apply_T(T2m, H), color="C1", label="T2")
ax.plot(*apply_T(T, H), color="C3", label="T1 T2")
draw_frame(ax, np.eye(3), name="W")
draw_frame(ax, T, name="body")
ax.legend()
plt.show()


## 5. 用 NumPy 对题 1–8

先自己算完再跑。基变换：`v = R @ v'` ⇒ `v' = R.T @ v`（R 为旋转）。


In [ ]:
print("题 1", np.array([3.0, 4.0]) @ np.array([4.0, -3.0]))

def cross2(u, v):
    return u[0] * v[1] - u[1] * v[0]

print("题 2", cross2((1, 0), (0, 1)), cross2((1, 0), (0, -1)), cross2((2, 0), (0, 3)))

R90 = rot2(np.pi / 2)
print("题 3 R@e1", R90 @ [1, 0], "R.T@R\n", R90.T @ R90, "det", np.linalg.det(R90))

for name, M in [("A", A), ("B", B), ("C", C), ("D", D)]:
    RtR = M.T @ M
    print(f"题 4 {name} det={np.linalg.det(M):.1f}  RTR≈I? {np.allclose(RtR, np.eye(2))}")

R = np.array([[1.0, -1.0], [1.0, 1.0]]) / np.sqrt(2)
v = np.array([1.0, 0.0])
print("题 7 v' =", R.T @ v)
